# Assumption 3: Exogeneity & Zero Conditional Mean ($E[\epsilon | X] = 0$)
**Prepared for:** Assumptions of Least Square Regressions Lecture Suite  
**Author:** Nick Lim

Under the Gauss-Markov framework, **Assumption 3** requires that the error term $\epsilon$ has an expected value of exactly zero given any value of the predictor variables ($X$):
$$E[\epsilon_i | X_i] = 0$$

### What causes violations of $E[\epsilon | X] = 0$?
1. **Omitted Variable Bias (OVB):** Leaving out a confounding variable $Z$ that is correlated with both the outcome $Y$ and the predictor $X$.
2. **Simultaneity / Endogeneity:** When $Y$ also causes $X$ in a feedback loop.
3. **Measurement Error in Predictors:** Attenuation bias caused by noisy measurement of $X$.

### What is the consequence?
When $E[\epsilon | X] \neq 0$, Ordinary Least Squares estimates ($\hat{\beta}_0, \hat{\beta}_1$) become **systematically biased and inconsistent**. Even if your sample size $n$ approaches infinity ($N \to \infty$), your OLS slope will **never converge to the true parameter**!

In this notebook, we simulate **Omitted Variable Bias (OVB)** across 1,000 points and run a Monte Carlo proof showing why endogeneity breaks OLS.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm

if not hasattr(matplotlib.rcParams, '_get'):
    matplotlib.rcParams._get = lambda key: matplotlib.rcParams[key]

np.random.seed(42)
sns.set_theme(style="whitegrid")
plt.rcParams["font.size"] = 12

## 1. Simulating Omitted Variable Bias ($N = 1,000$)

Suppose our true structural data-generating process depends on both $X$ (e.g., Years of Education) and an unobserved confounder $Z$ (e.g., Innate Ability / Parental Wealth):
$$Y = 5.0 + 2.0 X + 1.5 Z + u, \quad u \sim \mathcal{N}(0, 1)$$

Crucially, suppose $X$ and $Z$ are positively correlated ($Z = 0.8 X + v$). If a researcher only observes $X$ and fits the naive regression `Y ~ X`, the unobserved $1.5 Z$ gets forced into the error term:
$$\epsilon = 1.5 Z + u = 1.5 (0.8 X + v) + u = 1.2 X + \text{noise}$$

Because the error term $\epsilon$ now depends directly on $+1.2 X$, **$E[\epsilon | X] \neq 0$**! The OLS slope will be biased upward by exactly $+\beta_Z \times \frac{\text{Cov}(X, Z)}{\text{Var}(X)} = +1.5 \times 0.8 = +1.20$ (yielding $\hat{\beta}_1 \approx 3.20$ instead of $2.00$).

In [ ]:
N = 1000
x = np.random.uniform(0, 10, N)

# Confounder Z is correlated with X
z = 0.8 * x + np.random.normal(0, 1.0, N)

# True structural equation
y = 5.0 + 2.0 * x + 1.5 * z + np.random.normal(0, 1.0, N)

df = pd.DataFrame({"x": x, "z": z, "y": y})

# Model 1: Naive OLS omitting Z (Violating Assumption 3: Endogeneity)
mod_omitted = sm.OLS(df["y"], sm.add_constant(df["x"])).fit()

# Model 2: True Multiple OLS including Z (Restoring Exogeneity)
mod_full = sm.OLS(df["y"], sm.add_constant(df[["x", "z"]])).fit()

print("=== Model Comparison: Omitted Confounder vs. Full Exogenous Specification ===")
print(f"True Structural Slope for X : +2.000")
print(f"-> Naive Omitted Fit (y ~ x): +{mod_omitted.params['x']:.3f}  (Biased upward by +1.20 due to E[epsilon|X] != 0)")
print(f"-> Full Exogenous Fit       : +{mod_full.params['x']:.3f}  (Unbiased structural parameter recovery!)")

## 2. Visual Diagnostic: Error Expectations vs Predictor ($X$)

Let's visualize how violating $E[\epsilon | X] = 0$ distorts the regression line and creates a sloping relationship between the composite error and $X$.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))

# Left: Scatter plot comparing Omitted vs Full Slopes against True Partial Effect
axes[0].scatter(df["x"], df["y"], color="#cbd5e1", alpha=0.5, s=25, label="Observed Data ($Y$)")
xg = np.linspace(0, 10, 100)
axes[0].plot(xg, mod_omitted.predict(sm.add_constant(xg)), color="#dc2626", linewidth=3.2, linestyle="--", label=f"Naive Omitted Line ($\hat{{\beta}}_1 = {mod_omitted.params['x']:.2f}$)")
# Plot true partial relationship holding Z at mean
axes[0].plot(xg, 5.0 + 2.0 * xg + 1.5 * np.mean(z), color="#1e293b", linewidth=3.2, label="True Partial Slope ($\beta_1 = 2.00$)")
axes[0].set_title("Omitted Variable Bias: Distorting the Regression Slope")
axes[0].set_xlabel("Predictor ($X$)")
axes[0].set_ylabel("Outcome ($Y$)")
axes[0].legend()

# Right: Structural Error (epsilon = 1.5Z + u) vs X showing E[epsilon|X] != 0
composite_error = 1.5 * df["z"] + np.random.normal(0, 1.0, N)
axes[1].scatter(df["x"], composite_error, color="#f59e0b", alpha=0.5, s=25, label="Composite Error ($\epsilon = 1.5Z + u$)")
axes[1].axhline(0, color="#1e293b", linestyle="--", linewidth=1.5)
sns.regplot(x=df["x"], y=composite_error, scatter=False, color="#dc2626", ax=axes[1], line_kws={"linewidth": 3.0, "label": "$E[\epsilon | X] = 1.2X \neq 0$"})
axes[1].set_title("Violation of Assumption 3: $E[\epsilon | X]$ Slopes with $X$")
axes[1].set_xlabel("Predictor ($X$)")
axes[1].set_ylabel("True Structural Error ($\epsilon$)")
axes[1].legend()

plt.tight_layout()
plt.show()

## 3. Monte Carlo Simulation: Inconsistency ($N \to \infty$ Does Not Help)

Unlike sample noise (which shrinks to zero as $n \to \infty$), violating Assumption 3 introduces **asymptotic bias (inconsistency)**. Even if we increase our sample size to $N = 5,000$ across 500 simulations, the average estimated slope remains locked at `3.20` instead of `2.00`.

In [ ]:
sim_slopes = []
for _ in range(500):
    xs = np.random.uniform(0, 10, 5000)
    zs = 0.8 * xs + np.random.normal(0, 1.0, 5000)
    ys = 5.0 + 2.0 * xs + 1.5 * zs + np.random.normal(0, 1.0, 5000)
    ms = sm.OLS(ys, sm.add_constant(xs)).fit()
    sim_slopes.append(ms.params[1])

print(f"=== Monte Carlo Proof of Inconsistency across 500 Loops (N = 5,000 each) ===")
print(f"True Structural Slope          : 2.0000")
print(f"Mean Estimated Slope (mean(b1)): {np.mean(sim_slopes):.4f}")
print(f"-> Permanent Asymptotic Bias   : +{np.mean(sim_slopes) - 2.0:.4f}  (Will never vanish regardless of sample size!)")

### Key Takeaways
- **Exogeneity ($E[\epsilon|X]=0$) is the most critical Gauss-Markov assumption** for causal and structural interpretation.
- **Fixes:** If omitted confounders cannot be observed directly, economists and statisticians use **Instrumental Variables (`IV / 2SLS`)**, **Fixed Effects panel models**, or **Randomized Controlled Trials (`RCTs`)** to break the correlation between $X$ and $\epsilon$.